# 🦜🔗 LangChain Local AI — Practical Examples

> A hands-on notebook exploring LangChain with **local Ollama models** and **Google Gemini**.
> Covers chains, structured output, batch processing, image vision, agent memory, and human-in-the-loop.

---

## 📋 Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Basic Chain — Stream & Batch](#2-basic-chain--stream--batch)
3. [String Output Parser with Gemini](#3-string-output-parser-with-gemini)
4. [Structured Output with Pydantic](#4-structured-output-with-pydantic)
5. [Invoice Extraction (Single)](#5-invoice-extraction-single)
6. [Invoice Batch Processing](#6-invoice-batch-processing)
7. [Image Vision with Ollama](#7-image-vision-with-ollama)
8. [Multi-Step Pipeline with Gemini](#8-multi-step-pipeline-with-gemini)
9. [Agent Memory — Summarization Middleware](#9-agent-memory--summarization-middleware)
10. [Human-in-the-Loop Middleware](#10-human-in-the-loop-middleware)

---

## Requirements

- [Ollama](https://ollama.com/) installed and running locally
- Models pulled: `ollama pull llama3.2:1b` and `ollama pull moondream`
- A `.env` file containing:
  ```
  GOOGLE_API_KEY=your_key_here
  GEMINI_MODEL=gemini-2.0-flash
  OPENAI_API_KEY=your_key_here
  ```


---
## 1. Setup & Installation


In [1]:
# Install required packages (run once)
# !pip install langchain langchain-ollama langchain-google-genai langchain-openai \
#              langgraph pydantic python-dotenv

In [2]:
# --- Standard Library ---
import base64
import os

# --- Third-Party ---
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

# --- LangChain Core ---
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.tools import tool

# --- LangChain Integrations ---
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

# --- LangGraph ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()
print("✅ All imports loaded successfully.")

c:\Users\hp\Desktop\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports loaded successfully.


---
## 2. Basic Chain — Stream & Batch

Build a simple **prompt → LLM → parser** chain using LCEL (LangChain Expression Language).  
Demonstrates `.stream()` for real-time output and `.batch()` for parallel queries.


In [3]:
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0.3,  # Lower = more factual, less creative
)

prompt = ChatPromptTemplate.from_template("""
Answer the following question as a helpful assistant.
Question: {input}
Answer:""")

chain = prompt | llm | StrOutputParser()
print("✅ Chain ready.")

✅ Chain ready.


In [4]:
print("--- Streaming Response ---")
for chunk in chain.stream({"input": "Why do parrots talk?"}):
    print(chunk, end="", flush=True)

--- Streaming Response ---
Parrots are known for their ability to mimic human speech and other sounds, and they do so for several reasons. Here are some possible explanations:

1. **Social interaction**: Parrots are social birds that live in flocks, and they use vocalizations to communicate with each other. By mimicking human speech, they can interact with their human caregivers and other parrots in their flock.
2. **Learning and problem-solving**: Parrots are intelligent birds that are capable of learning and problem-solving. By mimicking human speech, they may be trying to learn new words or phrases, or to solve problems by figuring out how to use language to achieve a goal.
3. **Attention and affection**: Parrots are known to be highly social and attention-seeking birds. By mimicking human speech, they may be trying to get attention and affection from their caregivers.
4. **Self-expression**: Some parrots may simply enjoy mimicking human speech as a form of self-expression or play.


In [5]:
questions = [
    "What is the capital of France?",
    "What is the name of the largest and most intelligent mammal?",
]
responses = chain.batch(questions)
print("--- Batch Responses ---")
for q, r in zip(questions, responses):
    print(f"\nQ: {q}")
    print(f"A: {r}")

--- Batch Responses ---

Q: What is the capital of France?
A: The capital of France is Paris.

Q: What is the name of the largest and most intelligent mammal?
A: The largest and most intelligent mammal is the blue whale (Balaenoptera musculus). On average, an adult blue whale can weigh around 150-170 tons (136,000-152,000 kg) and reach lengths of up to 82 feet (25 meters). However, the largest blue whale ever recorded was a female that was found in 1947 off the coast of Iceland, which weighed around 210 tons (182,000 kg).

As for intelligence, blue whales are considered one of the most intelligent animals on Earth. They have been observed exhibiting complex behaviors such as:

* Social behavior: Blue whales have been known to form long-lasting social bonds with other whales.
* Communication: They use a variety of clicks, whistles, and moans to communicate with each other.
* Problem-solving: Blue whales have been observed using tools, such as sponges, to help them forage for food.
* Sel

---
## 3. String Output Parser with Gemini

Shows how `StrOutputParser` cleans raw LLM output into a plain string.  
Without the parser, Gemini returns a full `AIMessage` object including metadata.  
The third cell shows the cleanest pattern: composing it directly into an LCEL chain.


In [6]:
# Model name loaded from .env (e.g. "gemini-3.1-flash-lite-preview")
gemini_model_name = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")
llm_gemini = ChatGoogleGenerativeAI(
    model=gemini_model_name,
    temperature=0.6,
)
template = PromptTemplate.from_template("Who is the President of country {country}")
print("✅ Gemini LLM ready.")

✅ Gemini LLM ready.


In [7]:
# Without parser: raw AIMessage — includes metadata, signatures, etc.
chat = template.invoke({"country": "United States"})
raw_result = llm_gemini.invoke(chat)
print("--- Raw output type ---")
print(type(raw_result))
print("--- Text content ---")
print(raw_result.content)

--- Raw output type ---
<class 'langchain_core.messages.ai.AIMessage'>
--- Text content ---
[{'type': 'text', 'text': 'The current President of the United States is **Joe Biden**.', 'extras': {'signature': 'EjQKMgG+Pvb7FTI9Puy5ABxtLSwdT8mJJuAs1J7poPNv6xxo3ZuOui7b1X72wP1SN6Ep8MiW'}}]


In [8]:
# With parser: clean plain string
parser = StrOutputParser()
parsed_result = parser.invoke(raw_result)
print("--- Parsed output (with StrOutputParser) ---")
print(parsed_result)

--- Parsed output (with StrOutputParser) ---
The current President of the United States is **Joe Biden**.


In [9]:
# Best practice: compose into a single LCEL chain
gemini_chain = template | llm_gemini | parser
result = gemini_chain.invoke({"country": "India"})
print(result)

The current President of India is **Droupadi Murmu**. She assumed office on July 25, 2022.


---
## 4. Structured Output with Pydantic

Use `.with_structured_output()` to make the LLM return a validated **Pydantic object**  
instead of raw text — great for data extraction pipelines.


In [10]:
class ProductInfo(BaseModel):
    name: str      = Field(..., description="The name of the product")
    price: float   = Field(..., description="The price of the product in USD")
    category: str  = Field(..., description="The category of the product")
    in_stock: bool = Field(..., description="Whether the product is currently in stock")

product_llm = llm_gemini.with_structured_output(ProductInfo)
product = product_llm.invoke("What is the price and availability of the latest MacBook Pro 14?")

print(f"Product : {product.name}")
print(f"Price   : ${product.price}")
print(f"Category: {product.category}")
print(f"In Stock: {product.in_stock}")
print(f"\nJSON dump:\n{product.model_dump_json(indent=2)}")

Product : MacBook Pro 14-inch
Price   : $1599.0
Category: Electronics
In Stock: True

JSON dump:
{
  "name": "MacBook Pro 14-inch",
  "price": 1599.0,
  "category": "Electronics",
  "in_stock": true
}


---
## 5. Invoice Extraction (Single)

Extract structured data from raw invoice text using a **nested Pydantic schema**.  
Demonstrates `Literal` types, optional fields, and computed `@property` values.


In [ ]:
class LineItem(BaseModel):
    description: str
    quantity: int     = Field(ge=1)
    unit_price: float = Field(ge=0)

    @property
    def total(self) -> float:
        return self.quantity * self.unit_price


class Invoice(BaseModel):
    invoice_number: str
    date: str
    vendor_name: str
    vendor_address: Optional[str] = None
    items: List[LineItem]
    subtotal: float
    tax: Optional[float] = None
    total: float
    payment_status: Literal["paid", "pending", "overdue"] = "pending"


# json_schema is most reliable for complex nested schemas with local models
invoice_llm = ChatOllama(model="llama3.2:1b", temperature=0)
structured_invoice_llm = llm_gemini.with_structured_output(Invoice, method="json_schema")
print("✅ Invoice schema and LLM ready.")

✅ Invoice schema and LLM ready.


In [ ]:
invoice_text = """
Invoice #INV-2025-001
Date: January 15, 2025
From: Acme Corp, 123 Business St

Items:
- Widget Pro (5) @ $29.99 each
- Service Fee (1) @ $50.00

Subtotal: $199.95
Tax: $16.00
Total: $215.95

Status: Paid
"""
invoice = structured_invoice_llm.invoke(
    f"Extract all data from this invoice text accurately:\n{invoice_text}"
)
print(f"Invoice Number : {invoice.invoice_number}")
print(f"Vendor         : {invoice.vendor_name}")
print(f"Date           : {invoice.date}")
print(f"Vendor Address : {invoice.vendor_address or 'N/A'}")
print(f"Subtotal       : ${invoice.subtotal:.2f}")
print(f"Tax            : ${invoice.tax or 0:.2f}")
print(f"Total          : ${invoice.total:.2f}")
print(f"Status         : {invoice.payment_status}")
print("\nLine Items:")
for item in invoice.items:
    print(f"  - {item.description}: {item.quantity} x ${item.unit_price:.2f} = ${item.total:.2f}")

Invoice Number : INV-2025-001
Vendor         : Acme Corp
Date           : January 15, 2025
Vendor Address : N/A
Subtotal       : $199.95
Tax            : $16.00
Total          : $215.95
Status         : paid

Line Items:
  - Widget Pro (5): 5 x $29.99 = $149.95
  - Service Fee (1): 1 x $50.00 = $50.00


---
## 6. Invoice Batch Processing

Scale extraction to process **multiple invoices** in a loop with error handling.


In [13]:
class InvoiceSummary(BaseModel):
    invoice_number: str
    date: str
    vendor_name: str
    items: List[LineItem]
    total: float
    payment_status: Literal["paid", "pending", "overdue"] = "pending"

# we using the same gemini LLM, but you can choose a different one if you want

batch_structured_llm = llm_gemini.with_structured_output(InvoiceSummary, method="json_schema")

invoices_data = [
    "Invoice #INV-001, Date: Jan 10, From: Apple Store. Item: iPhone 15 (1) @ $799. Total: $799. Status: Paid",
    "Invoice #INV-002, Date: Jan 12, From: Amazon. Item: USB-C Cable (2) @ $15. Total: $30. Status: Pending",
    "Invoice #6789, Date: Feb 01, From: Cloud Hosting. Item: Server Pro (1) @ $50. Total: $50. Status: Overdue",
    "Invoice #TX-99, Date: Feb 05, From: Local Cafe. Item: Coffee (10) @ $4.50. Total: $45. Status: Paid",
]

processed_invoices = []
print(f"--- Processing {len(invoices_data)} invoices ---\n")
for i, text in enumerate(invoices_data, 1):
    try:
        print(f"Processing Invoice {i}...")
        result = batch_structured_llm.invoke(f"Extract structured data from this invoice text: {text}")
        processed_invoices.append(result)
    except Exception as e:
        print(f"  ⚠️  Error on invoice {i}: {e}")

print("\n--- Results ---")
STATUS_ICON = {"paid": "✅", "pending": "⏳", "overdue": "🔴"}
for inv in processed_invoices:
    icon = STATUS_ICON.get(inv.payment_status, "")
    print(f"[{inv.invoice_number}] {inv.vendor_name} — ${inv.total} {icon}")

--- Processing 4 invoices ---

Processing Invoice 1...
Processing Invoice 2...
Processing Invoice 3...
Processing Invoice 4...

--- Results ---
[INV-001] Apple Store — $799.0 ✅
[INV-002] Amazon — $30.0 ⏳
[6789] Cloud Hosting — $50.0 🔴
[TX-99] Local Cafe — $45.0 ✅


---
## 7. Image Vision with Ollama

Send images to a local vision model (`moondream`) using Base64 encoding.  
Works fully **offline** — no API key needed.

> 💡 Replace `IMAGE_PATH` with a real image path on your machine.


In [14]:
def encode_image(image_path: str) -> str:
    """Convert a local image file to a Base64-encoded string."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def analyze_image(image_path: str, prompt: str = "What is in this image?") -> str:
    """Send a local image to the Moondream vision model and return its description."""
    vision_llm = ChatOllama(model="moondream")
    image_b64  = encode_image(image_path)
    ext        = image_path.rsplit(".", 1)[-1].lower()
    mime       = "image/gif" if ext == "gif" else f"image/{ext}"

    message = HumanMessage(content=[
        {"type": "text",      "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{image_b64}"}},
    ])
    return vision_llm.invoke([message]).content


print("✅ Vision helper functions defined.")

✅ Vision helper functions defined.


In [15]:
# ✏️  Update this path to any image on your machine
IMAGE_PATH = "path/to/your/image.jpg"

if os.path.exists(IMAGE_PATH):
    description = analyze_image(
        IMAGE_PATH,
        prompt="According to what you see, can you explain what is happening in this image step by step?",
    )
    print("--- Model Analysis ---")
    print(description)
else:
    print(f"⚠️  Image not found at '{IMAGE_PATH}'. Update IMAGE_PATH to run this cell.")

⚠️  Image not found at 'path/to/your/image.jpg'. Update IMAGE_PATH to run this cell.


---
## 8. Multi-Step Pipeline with Gemini

A two-step chain powered by **Google Gemini**:
1. Ask the model to **pick** a well-known movie
2. Ask it to **detail** that movie using the first response as input

Showcases deeply nested schemas and chaining structured calls.

> 🔑 Requires `GOOGLE_API_KEY` in your `.env` file.


In [16]:
class MovieRef(BaseModel):
    title: str          = Field(..., description="Official movie title")
    year: Optional[int] = Field(None,  description="Release year if known")

class MoviePick(BaseModel):
    movie: MovieRef
    why_known: str    = Field(..., description="Short reason why this movie is well-known")
    confidence: float = Field(..., ge=0, le=1, description="Confidence score 0–1")

class PersonRole(BaseModel):
    name: str
    role: str = Field(..., description="Character name or production role")

class MovieMeta(BaseModel):
    genres: List[str]
    language: Optional[str] = None
    country: Optional[str]  = None
    runtime_minutes: Optional[int] = None

class MovieRatings(BaseModel):
    imdb: Optional[float] = Field(None, ge=0, le=10)
    rotten_tomatoes: Optional[int] = Field(None, ge=0, le=100)

class MovieDetails(BaseModel):
    movie: MovieRef
    plot: str
    themes: List[str]
    key_people: List[PersonRole]
    metadata: MovieMeta
    ratings: MovieRatings

print("✅ Movie schemas defined.")

✅ Movie schemas defined.


In [17]:
gemini = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-2.0-flash"),
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,
)

# Step 1: Pick a movie
pick_llm  = gemini.with_structured_output(MoviePick)
movie_pick = pick_llm.invoke(
    "Choose ONE real, famous movie you are very confident you know well. "
    "Return valid structured data only."
)

# Step 2: Get full details for that same movie
details_llm   = gemini.with_structured_output(MovieDetails)
movie_details  = details_llm.invoke(
    f"Provide accurate structured details for: {movie_pick.movie.title} ({movie_pick.movie.year})."
)

print(f"🎬 Picked: {movie_pick.movie.title} ({movie_pick.movie.year})")
print(f"   Confidence : {movie_pick.confidence:.0%}")
print(f"   Why known  : {movie_pick.why_known}")
print("\n--- MoviePick JSON ---")
print(movie_pick.model_dump_json(indent=2))
print("\n--- MovieDetails JSON ---")
print(movie_details.model_dump_json(indent=2))

🎬 Picked: The Godfather (1972)
   Confidence : 100%
   Why known  : Widely considered one of the greatest films in cinema history, directed by Francis Ford Coppola and starring Marlon Brando and Al Pacino.

--- MoviePick JSON ---
{
  "movie": {
    "title": "The Godfather",
    "year": 1972
  },
  "why_known": "Widely considered one of the greatest films in cinema history, directed by Francis Ford Coppola and starring Marlon Brando and Al Pacino.",
  "confidence": 1.0
}

--- MovieDetails JSON ---
{
  "movie": {
    "title": "The Godfather",
    "year": 1972
  },
  "plot": "The aging patriarch of an organized crime dynasty transfers control of his clandestine empire to his reluctant son.",
  "themes": [
    "Family",
    "Power",
    "Corruption",
    "Loyalty",
    "The American Dream"
  ],
  "key_people": [
    {
      "name": "Marlon Brando",
      "role": "Vito Corleone"
    },
    {
      "name": "Al Pacino",
      "role": "Michael Corleone"
    },
    {
      "name": "Francis Ford

---
## 9. Agent Memory — Summarization Middleware

As conversations grow, context windows fill up and costs increase.  
`SummarizationMiddleware` automatically compresses old messages into a summary  
once a trigger threshold is reached.

Two strategies:
- **Message-based** — summarize after N messages
- **Token-based** — summarize after N tokens (fraction of context window)


In [20]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver # 2026 Standard for InMemorySaver

load_dotenv()

# 1. Initialize the model using the Google AI Studio backend
# Ensure GOOGLE_API_KEY is in your .env file
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0
)

# 2. Configure the agent with Summarization Middleware
msg_agent = create_agent(
    model=llm,
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm, # Using the same model for summarization
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

msg_config = {"configurable": {"thread_id": "summarize-messages"}}
print("✅ Message-based summarization agent ready using AI Studio.")

✅ Message-based summarization agent ready using AI Studio.


In [22]:
for q in questions:
    response = msg_agent.invoke({"messages": [HumanMessage(content=q)]}, msg_config)
    
    # Extract the content and ensure it's a string for formatting
    raw_answer = response["messages"][-1].content
    answer = str(raw_answer) if isinstance(raw_answer, list) else raw_answer
    
    # Remove newlines for cleaner table printing
    answer = answer.replace('\n', ' ').strip()
    
    msg_count = len(response["messages"])
    print(f"{q:<22} {answer[:23]:<25} {msg_count}")

What is 2+2?           [{'type': 'text', 'text   4
What is 10*5?          [{'type': 'text', 'text   6
What is 100/4?         [{'type': 'text', 'text   8
What is 15-7?          [{'type': 'text', 'text   10
What is 3*3?           [{'type': 'text', 'text   6
What is 4*4?           [{'type': 'text', 'text   8


In [ ]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city — returns a formatted listing."""
    return (
        f"Hotels in {city}:\n"
        "  1. Grand Hotel  — 5★  $350/night  (spa, pool, gym)\n"
        "  2. City Inn     — 4★  $180/night  (business center)\n"
        "  3. Budget Stay  — 3★  $75/night   (free wifi)"
    )
# 1. Manually initialize the LLM to force the use of AI Studio / API Key
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

# 2. Pass the 'llm' instance (NOT a string) to create_agent
token_agent = create_agent(
    model=llm,  # Using the instance we created above
    tools=[search_hotels],
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,  # Use the same instance for summarization
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ],
)
token_config = {"configurable": {"thread_id": "summarize-tokens"}}


def count_tokens(messages: list) -> int:
    """Approximate token count (4 chars ≈ 1 token)."""
    return sum(len(str(m.content)) for m in messages) // 4

print("✅ Token-based summarization agent ready.")

✅ Token-based summarization agent ready.


In [28]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

print(f"{'City':<12} {'~Tokens':>8}  {'Messages':>10}")
print("-" * 35)
for city in cities:
    response = token_agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=token_config,
    )
    tokens = count_tokens(response["messages"])
    msgs   = len(response["messages"])
    print(f"{city:<12} {tokens:>8}  {msgs:>10}")

City          ~Tokens    Messages
-----------------------------------
Paris             136           4
London            272           8
Tokyo             408          12
New York          454           9
Dubai             464           9
Singapore         471           9


---
## 10. Human-in-the-Loop Middleware

Pause agent execution before a tool runs, then let a human **approve**, **reject**, or **edit** the call.  
Critical for high-stakes operations — sending emails, database writes, financial transactions.

Three scenarios:
- **10a. Approve** — let the action proceed as-is
- **10b. Reject** — cancel the action
- **10c. Edit** — modify arguments before execution


In [37]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

# --- Shared mock tools ---

@tool
def read_email(email_id: str) -> str:
    """Read an email by its ID."""
    return f"Email content for ID: {email_id}"


@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {recipient} with subject '{subject}'"


def build_hitl_agent(thread_id: str):
    agent = create_agent(
        model=llm,
        tools=[read_email, send_email],
        checkpointer=InMemorySaver(),
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
                    "read_email": False,  # No human review for reads
                }
            )
        ],
    )
    config = {"configurable": {"thread_id": thread_id}}
    return agent, config

EMAIL_REQUEST = "Send email to john@test.com with subject 'Hello' and body 'How are you?'"
print("✅ HITL agent factory ready.")

✅ HITL agent factory ready.


### 10a. Approve — let the tool call execute as-is


In [38]:
agent, config = build_hitl_agent("hitl-approve")

# Step 1: Agent pauses before executing send_email
result = agent.invoke({"messages": [HumanMessage(content=EMAIL_REQUEST)]}, config=config)

if "__interrupt__" in result:
    print("⏸️  Agent paused — awaiting human approval...")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️  Agent paused — awaiting human approval...
✅ Result: [{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EjQKMgG+Pvb7f1BCSmj2r4jNDjU5GM3fWsVEq96BmYaY/XwZGcaR9FJDEpHmg2jJnZfB5tEA'}}]


### 10b. Reject — cancel the tool call


In [39]:
agent, config = build_hitl_agent("hitl-reject")

result = agent.invoke({"messages": [HumanMessage(content=EMAIL_REQUEST)]}, config=config)

if "__interrupt__" in result:
    print("⏸️  Agent paused — rejecting the tool call...")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config,
    )
    print(f"🚫 Result: {result['messages'][-1].content}")

⏸️  Agent paused — rejecting the tool call...
🚫 Result: [{'type': 'text', 'text': 'The request to send the email was rejected. Please let me know if you would like me to try again or if there is anything else I can help you with.', 'extras': {'signature': 'EjQKMgG+Pvb7RyYW966gAM8y2RCh68zzxKzuP0XTyTHzsCvrJJzcKR8CMw/IDYOdPjP1uhfT'}}]


### 10c. Edit — modify the arguments before the tool executes


In [41]:
agent, config = build_hitl_agent("hitl-edit")

result = agent.invoke(
    {"messages": [HumanMessage(content="Send an email to boss@company.com about the report")]},
    config=config,
)
if "__interrupt__" in result:
    print("⏸️  Agent paused — editing arguments before sending...")
    
    # Correct schema for 'edit' decision
    result = agent.invoke(
        Command(resume={
            "decisions": [{
                "type": "edit",
                "edited_action": {  # <--- This wrapper is required
                    "name": "send_email", # Specify the tool name
                    "args": {
                        "recipient": "boss@company.com",
                        "subject":   "Q1 Report — Final",
                        "body":      "Hi, please find the final Q1 report attached.",
                    }
                }
            }]
        }),
        config=config,
    )
    print(f"✏️  Result: {result['messages'][-1].content}")

⏸️  Agent paused — editing arguments before sending...
✏️  Result: [{'type': 'text', 'text': "OK. I've sent the email to your boss regarding the report.", 'extras': {'signature': 'EjQKMgG+Pvb7/mIbYju3VhrH0NB8iFX586XtGLk2jLxGYo/2ztByoRSqG6Wt1hrgg/30fa/o'}}]


# Document Loaders

## Text Loader

In [42]:
from langchain_community.document_loaders import TextLoader
path = "pyproject.toml"
loader = TextLoader(path)
print(f"✅ Loaded document from {path}:\n")
print(loader.load()[0].page_content)

✅ Loaded document from pyproject.toml:

[project]
name = "langchain-app"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.12"
dependencies = [
    "jq>=1.11.0",
    "langchain>=1.2.12",
    "langchain-chroma>=1.1.0",
    "langchain-community>=0.4.1",
    "langchain-google-genai>=4.2.1",
    "langchain-google-vertexai>=3.2.2",
    "langchain-ollama>=1.0.1",
    "langchain-text-splitters>=1.1.1",
    "pydantic>=2.12.5",
    "pydantic-settings>=2.13.1",
    "pypdf>=6.9.2",
    "python-dotenv>=1.2.2",
]



## PyPdf Loader

In [43]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "data/Machine Learning.pdf"  # Update with your PDF file path
try:
    pdf_loader = PyPDFLoader(pdf_path)
    print(f"✅ Loaded PDF document from {pdf_path}:\n")
    docs = pdf_loader.load()
    print(f"Document has {len(docs)} pages.")
    print(docs[0].page_content)
except Exception as e:
    print(f"❌ Error occurred while loading PDF document | {e}")

✅ Loaded PDF document from data/Machine Learning.pdf:

Document has 392 pages.
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


In [44]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

directory_path = "data\pdf_file"  # Update with your directory path

pdf_directory_loader = PyPDFDirectoryLoader(directory_path , glob="**/*.pdf")

docs = pdf_directory_loader.load()

print(f"Document has {len(docs)} pages.")
print(docs[0].page_content)

<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\hp\AppData\Local\Temp\ipykernel_17676\2007544838.py:3: SyntaxWarning: invalid escape sequence '\p'
  directory_path = "data\pdf_file"  # Update with your directory path


Document has 784 pages.
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


## CSV Loader

In [45]:
from langchain_community.document_loaders import CSVLoader

path = "data/ecomm.csv"

csv_loader = CSVLoader(path)

csv_docs= csv_loader.load()

print(len(csv_docs))
print(csv_docs[20].page_content)

418
InvoiceNo: 536367
StockCode: 48187
Description: DOORMAT NEW ENGLAND
Quantity: 4
InvoiceDate: 12-01-2010 08:34
UnitPrice: 7.95
CustomerID: 13047
Country: United Kingdom


## CSV Loader

In [46]:
from langchain_community.document_loaders import CSVLoader

path = "data/ecomm.csv"

csv_loader = CSVLoader(path)

csv_docs= csv_loader.load()

print(len(csv_docs))
print(csv_docs[20].page_content)

418
InvoiceNo: 536367
StockCode: 48187
Description: DOORMAT NEW ENGLAND
Quantity: 4
InvoiceDate: 12-01-2010 08:34
UnitPrice: 7.95
CustomerID: 13047
Country: United Kingdom


## JSON Loader

In [47]:
from langchain_community.document_loaders import JSONLoader

path = "data/sample.json"

json_laoder = JSONLoader(path, jq_schema=".", text_content=False)

json_docs = json_laoder.load()

print(len(json_docs))
print(json_docs[0].page_content)

1
{"name": "Alice", "age": 25, "skills": ["Python", "LangChain", "Data Analysis"], "is_active": true}


## Structured File Loader

In [48]:
from langchain_community.document_loaders import (
    JSONLoader, 
    PyPDFLoader, 
    TextLoader, 
    CSVLoader
)
from pathlib import Path

# Required dependencies: uv add jq pypdf
files = Path("data").glob("*")
all_docs = []

for file in files:
    try:
        if file.suffix == ".json":
            loader = JSONLoader(str(file), jq_schema=".", text_content=False)
        elif file.suffix == ".pdf":
            loader = PyPDFLoader(str(file))
        elif file.suffix == ".csv":
            loader = CSVLoader(str(file))
        elif file.suffix in [".txt", ".md"]:
            loader = TextLoader(str(file))
        else:
            print(f"Skipping unsupported file type: {file.name}")
            continue
            
        all_docs.extend(loader.load())
    except Exception as e:
        print(f"Failed to load {file.name}: {e}")

print(f"Total documents loaded: {len(all_docs)}")

Skipping unsupported file type: pdf_file
Total documents loaded: 819


In [49]:
print(all_docs[65].page_content)

InvoiceNo: 536373
StockCode: 82482
Description: WOODEN PICTURE FRAME WHITE FINISH
Quantity: 6
InvoiceDate: 12-01-2010 09:02
UnitPrice: 2.1
CustomerID: 17850
Country: United Kingdom


---

## ✅ Summary

| Section | Concept Covered |
|---|---|
| Basic Chain | LCEL, `.stream()`, `.batch()` |
| String Output Parser | `PromptTemplate`, `StrOutputParser`, Gemini |
| Structured Output | Pydantic + `.with_structured_output()` |
| Invoice Extraction | Nested schemas, `Literal`, `Optional`, `@property` |
| Batch Processing | Loops, error handling, `format="json"` |
| Image Vision | Base64 encoding, multimodal `HumanMessage` |
| Multi-Step Gemini | Chaining structured calls across two LLM steps |
| Agent Memory | `SummarizationMiddleware` — message & token triggers |
| Human-in-the-Loop | `HumanInTheLoopMiddleware` — approve / reject / edit |

---
*Built with [LangChain](https://python.langchain.com/) · [Ollama](https://ollama.com/) · [Google Gemini](https://ai.google.dev/) · [OpenAI](https://platform.openai.com/)*
